In [ ]:
# Install required packages
%pip install tensorflow numpy matplotlib seaborn scikit-learn pandas

# Vanilla RNN Model with TensorFlow/Keras

This notebook implements a Vanilla RNN (Recurrent Neural Network) using TensorFlow/Keras for sentiment analysis on the IMDB dataset.

## Objectives:
- Load and preprocess the IMDB dataset
- Build the Vanilla RNN model architecture
- Train with different optimizers (SGD, Adam, RMSProp)
- Apply regularization techniques (Dropout, L1/L2, EarlyStopping)
- Perform basic hyperparameter tuning
- Visualize training curves and confusion matrix
- Evaluate model performance
- Save the trained model

In [ ]:
# Import Required Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd
import os
from collections import Counter

In [ ]:
# Dataset Loading Function
def load_imdb_dataset():
    """
    Load IMDB dataset from TensorFlow/Keras
    """
    print("Loading IMDB dataset from TensorFlow/Keras...")
    # Load IMDB dataset
    (train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(num_words=10000)
    return (train_data, train_labels), (test_data, test_labels)

# Load the dataset
(train_data, train_labels), (test_data, test_labels) = load_imdb_dataset()
print("Dataset loaded successfully!")

In [ ]:
# Dataset Exploration
def explore_imdb_dataset(train_data, train_labels, test_data, test_labels):
    """
    Display dataset shape, sample data, and label distribution
    """
    print("Dataset Info:")
    print(f"Train data shape: {len(train_data)}")
    print(f"Train labels shape: {len(train_labels)}")
    print(f"Test data shape: {len(test_data)}")
    print(f"Test labels shape: {len(test_labels)}")
    
    # Get word index
    word_index = keras.datasets.imdb.get_word_index()
    reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
    
    # Sample data
    print("\nSample Data:")
    for i in range(3):
        decoded_review = ' '.join([reverse_word_index.get(i - 3, '?') for i in train_data[i]])
        print(f"Review {i+1}: {decoded_review[:200]}...")
        print(f"Label: {'Positive' if train_labels[i] == 1 else 'Negative'}")
        print("-" * 50)
    
    # Label distribution
    train_label_counts = Counter(train_labels)
    test_label_counts = Counter(test_labels)
    
    print("\nLabel Distribution:")
    print("Train set:")
    print(f"  Negative (0): {train_label_counts[0]}")
    print(f"  Positive (1): {train_label_counts[1]}")
    
    print("Test set:")
    print(f"  Negative (0): {test_label_counts[0]}")
    print(f"  Positive (1): {test_label_counts[1]}")
    
    # Plot distribution
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    labels = ['Negative', 'Positive']
    train_counts = [train_label_counts[0], train_label_counts[1]]
    test_counts = [test_label_counts[0], test_label_counts[1]]
    
    ax1.bar(labels, train_counts)
    ax1.set_title('Training Set Class Distribution')
    ax1.set_ylabel('Count')
    
    ax2.bar(labels, test_counts)
    ax2.set_title('Test Set Class Distribution')
    ax2.set_ylabel('Count')
    
    plt.show()
    
    # Sequence length distribution
    train_lengths = [len(seq) for seq in train_data]
    test_lengths = [len(seq) for seq in test_data]
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.hist(train_lengths, bins=50, alpha=0.7)
    plt.title('Training Sequence Length Distribution')
    plt.xlabel('Sequence Length')
    plt.ylabel('Frequency')
    
    plt.subplot(1, 2, 2)
    plt.hist(test_lengths, bins=50, alpha=0.7)
    plt.title('Test Sequence Length Distribution')
    plt.xlabel('Sequence Length')
    plt.ylabel('Frequency')
    
    plt.show()

# Explore the dataset
explore_imdb_dataset(train_data, train_labels, test_data, test_labels)

In [ ]:
# Preprocessing
def preprocess_imdb_data(train_data, train_labels, test_data, test_labels, max_len=500):
    """
    Preprocess IMDB data: pad sequences, split validation
    """
    # Pad sequences to fixed length
    train_data = keras.preprocessing.sequence.pad_sequences(train_data, maxlen=max_len)
    test_data = keras.preprocessing.sequence.pad_sequences(test_data, maxlen=max_len)
    
    # Split train into train and validation
    val_split = 0.1
    val_size = int(len(train_data) * val_split)
    
    val_data = train_data[:val_size]
    val_labels = train_labels[:val_size]
    train_data = train_data[val_size:]
    train_labels = train_labels[val_size:]
    
    return (train_data, train_labels), (val_data, val_labels), (test_data, test_labels)

# Preprocess data
(train_X, train_y), (val_X, val_y), (test_X, test_y) = preprocess_imdb_data(
    train_data, train_labels, test_data, test_labels
)

print(f"Training data shape: {train_X.shape}")
print(f"Validation data shape: {val_X.shape}")
print(f"Test data shape: {test_X.shape}")
print(f"Number of classes: {len(np.unique(train_y))}")

In [ ]:
# Build Vanilla RNN Model
def build_vanilla_rnn_model(vocab_size=10000, embedding_dim=32, rnn_units=32, num_classes=1):
    """
    Build Vanilla RNN model with regularization
    """
    model = keras.Sequential([
        layers.Embedding(vocab_size, embedding_dim, input_length=train_X.shape[1]),
        layers.SimpleRNN(rnn_units, kernel_regularizer=keras.regularizers.l2(0.01),
                        recurrent_regularizer=keras.regularizers.l2(0.01)),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l1_l2(l1=0.01, l2=0.01)),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='sigmoid')
    ])
    
    return model

# Build model
model = build_vanilla_rnn_model()

# Print model summary
model.summary()

In [ ]:
# Training with Different Optimizers
def train_with_optimizer(model, optimizer_name, train_X, train_y, val_X, val_y, epochs=10):
    """
    Train model with specified optimizer and regularization
    """
    if optimizer_name == 'SGD':
        optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
    elif optimizer_name == 'Adam':
        optimizer = keras.optimizers.Adam(learning_rate=0.001)
    elif optimizer_name == 'RMSProp':
        optimizer = keras.optimizers.RMSprop(learning_rate=0.001)
    
    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    
    # Early stopping
    early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    history = model.fit(train_X, train_y,
                       validation_data=(val_X, val_y),
                       epochs=epochs,
                       batch_size=32,
                       callbacks=[early_stopping],
                       verbose=1)
    
    return history

# Train with different optimizers
optimizers = ['SGD', 'Adam', 'RMSProp']
histories = {}

for opt in optimizers:
    print(f"\nTraining with {opt} optimizer...")
    model_copy = build_vanilla_rnn_model()
    history = train_with_optimizer(model_copy, opt, train_X, train_y, val_X, val_y)
    histories[opt] = history
    print(f"{opt} training completed.")

In [ ]:
# Hyperparameter Tuning (Basic)
def hyperparameter_tuning(train_X, train_y, val_X, val_y):
    """
    Basic hyperparameter tuning for embedding dimension and RNN units
    """
    best_model = None
    best_val_acc = 0
    best_params = {}
    
    embedding_dims = [16, 32]
    rnn_units_list = [32, 64]
    
    for emb_dim in embedding_dims:
        for rnn_units in rnn_units_list:
            print(f"\nTuning: embedding_dim={emb_dim}, rnn_units={rnn_units}")
            
            model = build_vanilla_rnn_model(embedding_dim=emb_dim, rnn_units=rnn_units)
            model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                         loss='binary_crossentropy',
                         metrics=['accuracy'])
            
            early_stopping = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True)
            
            history = model.fit(train_X, train_y,
                               validation_data=(val_X, val_y),
                               epochs=5,
                               batch_size=32,
                               callbacks=[early_stopping],
                               verbose=0)
            
            val_acc = max(history.history['val_accuracy'])
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model = model
                best_params = {'embedding_dim': emb_dim, 'rnn_units': rnn_units}
    
    print(f"\nBest hyperparameters: {best_params}")
    print(f"Best validation accuracy: {best_val_acc}")
    
    return best_model, best_params

# Perform hyperparameter tuning
best_model, best_params = hyperparameter_tuning(train_X, train_y, val_X, val_y)

In [ ]:
# Plot Training Curves
def plot_training_curves(histories):
    """
    Plot training vs validation loss and accuracy for different optimizers
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    for i, (opt, history) in enumerate(histories.items()):
        # Loss
        axes[0, i].plot(history.history['loss'], label='Training Loss')
        axes[0, i].plot(history.history['val_loss'], label='Validation Loss')
        axes[0, i].set_title(f'{opt} - Loss')
        axes[0, i].set_xlabel('Epoch')
        axes[0, i].set_ylabel('Loss')
        axes[0, i].legend()
        
        # Accuracy
        axes[1, i].plot(history.history['accuracy'], label='Training Accuracy')
        axes[1, i].plot(history.history['val_accuracy'], label='Validation Accuracy')
        axes[1, i].set_title(f'{opt} - Accuracy')
        axes[1, i].set_xlabel('Epoch')
        axes[1, i].set_ylabel('Accuracy')
        axes[1, i].legend()
    
    plt.tight_layout()
    plt.show()

# Plot curves for different optimizers
plot_training_curves(histories)

In [ ]:
# Model Evaluation
def evaluate_model(model, test_X, test_y):
    """
    Evaluate model performance with accuracy, loss, F1 score, and confusion matrix
    """
    # Predictions
    predictions = model.predict(test_X)
    pred_labels = (predictions > 0.5).astype(int).flatten()
    
    # Calculate metrics
    accuracy = np.mean(pred_labels == test_y)
    loss = model.evaluate(test_X, test_y, verbose=0)[0]
    
    # F1 Score (macro average)
    f1 = f1_score(test_y, pred_labels, average='macro')
    
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Loss: {loss:.4f}")
    print(f"F1 Score (Macro): {f1:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(test_y, pred_labels, target_names=['Negative', 'Positive']))
    
    # Confusion Matrix
    cm = confusion_matrix(test_y, pred_labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    
    return accuracy, loss, f1

# Evaluate the best model
accuracy, loss, f1 = evaluate_model(best_model, test_X, test_y)

In [ ]:
# Save Model
def save_model(model, filename='vanilla_rnn_model.h5'):
    """
    Save the trained model
    """
    model.save(filename)
    print(f"Model saved as {filename}")

# Save the best model
save_model(best_model, 'vanilla_rnn_model.h5')

## Summary

This notebook implemented a Vanilla RNN (Recurrent Neural Network) using TensorFlow/Keras for IMDB sentiment analysis:

1. **Dataset**: Loaded IMDB dataset from TensorFlow/Keras
2. **Preprocessing**: Padded sequences to fixed length, split validation set
3. **Model**: Vanilla RNN with SimpleRNN layers, embedding layer
4. **Training**: Tested SGD, Adam, and RMSProp optimizers
5. **Regularization**: Applied L1/L2 regularization, Dropout, and Early Stopping
6. **Tuning**: Basic hyperparameter tuning for embedding dimension and RNN units
7. **Visualization**: Training curves and confusion matrix
8. **Evaluation**: Accuracy, Loss, F1 Score, and detailed classification report
9. **Model Saving**: Saved as .h5 file

The Vanilla RNN model provides a baseline for sequence modeling. You can further improve it by:
- Using more advanced RNN variants (LSTM, GRU)
- Adding bidirectional layers
- Implementing attention mechanisms
- Using pre-trained embeddings (GloVe, Word2Vec)
- Adding more RNN layers or different architectures
- Experimenting with different sequence lengths and vocabulary sizes